<a href="https://colab.research.google.com/github/Qsilver-ops/AI-SocialMedia-ContentModeration/blob/main/Group7_BERT_vs_BERTweet_Sentiment_Project_CS561.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

import re
import gc
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# LABELS
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

# DATASET
train_url = "https://raw.githubusercontent.com/cblancac/SentimentAnalysisBert/main/data/train_150k.txt"
test_url = "https://raw.githubusercontent.com/cblancac/SentimentAnalysisBert/main/data/test_62k.txt"

# loads raw files directly instead of using Python dataset script
raw = load_dataset(
    "text",
    data_files={
        "train": train_url,
        "test": test_url
    }
)

# turn each line into text + label
def parse_line(example):
    parts = example["text"].split("\t", 1)
    return {
        "text": parts[1].strip(),
        "labels": int(parts[0])
    }

raw = raw.map(parse_line, remove_columns=["text"])

# this makes validation split from train like the dataset script does
train_valid = raw["train"].train_test_split(test_size=0.2, seed=42)
ds = DatasetDict({
    "train": train_valid["train"],
    "validation": train_valid["test"],
    "test": raw["test"]
})

# use full dataset for final run
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

# print out dataset structure, first training example, and size of each split
print(ds)
print(train_ds[0])
print("train size:", len(train_ds))
print("val size:", len(val_ds))
print("test size:", len(test_ds))

# print out number of labels in training dataset to check for class imbalance
labels = train_ds["labels"]
print("negative (0):", labels.count(0))
print("positive (1):", labels.count(1))

# PREPROCESSING HELPER
def preprocess_text(text):
    text = re.sub(r"http\S+", "[URL]", text)
    text = re.sub(r"@\w+", "[USER]", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

# METRICS HELPER
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# SAMPLE PREDICTIONS HELPER
def show_samples(trainer, test_ds, test_tok, id2label, model_name, num_samples=3):
    sample_tok = test_tok.select(range(num_samples))
    sample_ds = test_ds.select(range(num_samples))
    pred_output = trainer.predict(sample_tok)
    preds = np.argmax(pred_output.predictions, axis=1)
    df = pd.DataFrame({
        "text": sample_ds["text"],
        "true_label": [id2label[x] for x in sample_ds["labels"]],
        "pred_label": [id2label[x] for x in preds],
    })
    print(f"\n{model_name} sample predictions:")
    print(df.to_string(index=False))

# TOKEN PREVIEW HELPER
def show_token_preview(tokenizer, text, model_name):
    cleaned = preprocess_text(text)
    tokens = tokenizer.tokenize(cleaned)[:20]
    print(f"\n{model_name} token preview:")
    print("original text:", text)
    print("cleaned text:", cleaned)
    print("tokens:", tokens)

# common speed settings
use_fp16 = torch.cuda.is_available()
train_bs = 16
eval_bs = 32
max_len = 128

# ══════════════════════════════════════════════════════════════════════════════
# 1. BERT
# ══════════════════════════════════════════════════════════════════════════════
bert_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased") # Huggingface BERT model

def bert_tokenize(batch):
    cleaned = [preprocess_text(t) for t in batch["text"]]
    return bert_tokenizer(cleaned, truncation=True, max_length=max_len)

show_token_preview(bert_tokenizer, test_ds[0]["text"], "BERT")

# tokenize all 3 dataset splits
train_tok_bert = train_ds.map(bert_tokenize, batched=True)
val_tok_bert   = val_ds.map(bert_tokenize,   batched=True)
test_tok_bert  = test_ds.map(bert_tokenize,  batched=True)

# load pretrained BERT weights and attach a 2-class classification head on top
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2, id2label=id2label, label2id=label2id
)

# fine-tune model
bert_trainer = Trainer(
    model=bert_model,
    args=TrainingArguments(
        output_dir="bert_run",
        eval_strategy="no", save_strategy="no",
        learning_rate=2e-5, num_train_epochs=1,
        per_device_train_batch_size=train_bs, per_device_eval_batch_size=eval_bs,
        weight_decay=0.01, logging_strategy="no", report_to="none",
        fp16=use_fp16
    ),
    train_dataset=train_tok_bert,
    eval_dataset=test_tok_bert,
    data_collator=DataCollatorWithPadding(tokenizer=bert_tokenizer),
    compute_metrics=compute_metrics
)

# perform fine-tuning and evaluate against test set
bert_trainer.train()
bert_results = bert_trainer.evaluate(test_tok_bert)
print("\nBERT results:", bert_results)
show_samples(bert_trainer, test_ds, test_tok_bert, id2label, "BERT")

# free memory after final model for faster execution
del bert_model, bert_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════════════════
# 2. RoBERTa
# ══════════════════════════════════════════════════════════════════════════════
roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base") # Huggingface RoBERTa model

def roberta_tokenize(batch):
    cleaned = [preprocess_text(t) for t in batch["text"]]
    return roberta_tokenizer(cleaned, truncation=True, max_length=max_len)

show_token_preview(roberta_tokenizer, test_ds[0]["text"], "RoBERTa")

# tokenize all 3 dataset splits
train_tok_roberta = train_ds.map(roberta_tokenize, batched=True)
val_tok_roberta   = val_ds.map(roberta_tokenize,   batched=True)
test_tok_roberta  = test_ds.map(roberta_tokenize,  batched=True)

# load pretrained RoBERTa weights with 2-class classification head
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=2, id2label=id2label, label2id=label2id
)

# fine-tune model
roberta_trainer = Trainer(
    model=roberta_model,
    args=TrainingArguments(
        output_dir="roberta_run",
        eval_strategy="no", save_strategy="no",
        learning_rate=2e-5, num_train_epochs=1,
        per_device_train_batch_size=train_bs, per_device_eval_batch_size=eval_bs,
        weight_decay=0.01, logging_strategy="no", report_to="none",
        fp16=use_fp16
    ),
    train_dataset=train_tok_roberta,
    eval_dataset=test_tok_roberta,
    data_collator=DataCollatorWithPadding(tokenizer=roberta_tokenizer),
    compute_metrics=compute_metrics
)

# perform fine-tuning on the sentiment training set and evaluate against test set
roberta_trainer.train()
roberta_results = roberta_trainer.evaluate(test_tok_roberta)
print("\nRoBERTa results:", roberta_results)
show_samples(roberta_trainer, test_ds, test_tok_roberta, id2label, "RoBERTa")

# free memory after final model for faster execution
del roberta_model, roberta_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════════════════
# 3. BERTweet
# ══════════════════════════════════════════════════════════════════════════════
bertweet_tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=False) # Huggingface BERTweet model, uses Python-based tokenizer, so it can run without error

def bertweet_tokenize(batch):
    cleaned = [preprocess_text(t) for t in batch["text"]]
    return bertweet_tokenizer(cleaned, truncation=True, max_length=max_len)

show_token_preview(bertweet_tokenizer, test_ds[0]["text"], "BERTweet")

# tokenize all 3 dataset splits
train_tok_bertweet = train_ds.map(bertweet_tokenize, batched=True)
val_tok_bertweet   = val_ds.map(bertweet_tokenize,   batched=True)
test_tok_bertweet  = test_ds.map(bertweet_tokenize,  batched=True)

# load pretrained BERTweet weights with 2-class classification head
bertweet_model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=2, id2label=id2label, label2id=label2id
)

# fine-tune model
bertweet_trainer = Trainer(
    model=bertweet_model,
    args=TrainingArguments(
        output_dir="bertweet_run",
        eval_strategy="no", save_strategy="no",
        learning_rate=2e-5, num_train_epochs=1,
        per_device_train_batch_size=train_bs, per_device_eval_batch_size=eval_bs,
        weight_decay=0.01, logging_strategy="no", report_to="none",
        fp16=use_fp16
    ),
    train_dataset=train_tok_bertweet,
    eval_dataset=test_tok_bertweet,
    data_collator=DataCollatorWithPadding(tokenizer=bertweet_tokenizer),
    compute_metrics=compute_metrics
)

# perform fine-tuning against sentiment training set and evaluate against test set
bertweet_trainer.train()
bertweet_results = bertweet_trainer.evaluate(test_tok_bertweet)
print("\nBERTweet results:", bertweet_results)
show_samples(bertweet_trainer, test_ds, test_tok_bertweet, id2label, "BERTweet")

# free memory after final model for faster execution
del bertweet_model, bertweet_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════════════════
# 4. Comparison
# ══════════════════════════════════════════════════════════════════════════════
comparison = pd.DataFrame({
    "model":     ["BERT", "RoBERTa", "BERTweet"],
    "accuracy":  [bert_results["eval_accuracy"],  roberta_results["eval_accuracy"],  bertweet_results["eval_accuracy"]],
    "precision": [bert_results["eval_precision"], roberta_results["eval_precision"], bertweet_results["eval_precision"]],
    "recall":    [bert_results["eval_recall"],    roberta_results["eval_recall"],    bertweet_results["eval_recall"]],
    "f1":        [bert_results["eval_f1"],        roberta_results["eval_f1"],        bertweet_results["eval_f1"]],
})
print("\nModel comparison:")
print(comparison.to_string(index=False))

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/149985 [00:00<?, ? examples/s]

Map:   0%|          | 0/61998 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 119988
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 29997
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 61998
    })
})
{'text': '@fa6ami86 so happy that salman won.  btw the 14sec clip is truely a teaser', 'labels': 0}
train size: 119988
val size: 29997
test size: 61998
negative (0): 59969
positive (1): 60019


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Step,Training Loss



BERTweet results: {'eval_loss': 0.30509424209594727, 'eval_accuracy': 0.8754153359785799, 'eval_precision': 0.873489534920991, 'eval_recall': 0.878275161945277, 'eval_f1': 0.8758758115317863, 'eval_runtime': 42.5693, 'eval_samples_per_second': 1456.401, 'eval_steps_per_second': 45.526, 'epoch': 1.0}

BERTweet sample predictions:
                                                                                                                                 text true_label pred_label
                                                                                 @justineville ...yeahhh. ) i'm 39 tweets from 1,600!   positive   positive
                                                                          @ApplesnFeathers aww. Poor baby! On your only REAL day off.   negative   negative
@joeymcintyre With my refunded $225 (Australian ticket price) I bought me a hot pair of brown boots  Woulda rathered seeing U any day   negative   positive

Model comparison:
   model  accuracy  preci

First, we load the dataset directly from raw .txt files hosted on GitHub. Each line in the file follows the format "0\ttweet text", (a sentiment label and tweet separated by a tab) so we parse each line to extract the label as an integer and the text as a string. Since the raw data only provides train and test splits, we carve 20% off the training set to create a validation split, giving us three splits: training, validation, and test.

We then preprocess the raw tweet text before passing it to any model. This involves replacing URLs with [URL], @mentions with [USER], stripping the # symbol from hashtags, collapsing extra whitespace, and lowercasing the text. Rather than removing URLs and mentions entirely, we replace them with placeholder tokens to preserve positional information: the model can still learn that a mention or link was present, which can carry sentiment signal.
Each model has its own tokenizer which converts the cleaned text into token IDs and attention masks the model can process. Tokenization differs meaningfully across the three models: BERT uses WordPiece tokenization trained on Wikipedia and BookCorpus, which causes informal words like "omg" or "lmao" to be fragmented into meaningless subword pieces. RoBERTa uses byte-pair encoding and handles some informal language better. BERTweet uses a tokenizer built from ~850 million tweets, so slang, abbreviations, and informal spellings are recognized as proper tokens rather than being broken apart.

We load three pretrained models (BERT, RoBERTa, and BERTweet) each with a classification head added on top that maps the model's output to our two sentiment classes (positive and negative). Then, we fine-tune each model on the training set. This is done through standard gradient descent where on each training step, the model makes predictions, cross-entropy loss is computed against the true labels, and backpropagation updates the weights in the direction that reduces the loss.

After fine-tuning, we evaluate each model on the held-out test set and collect four metrics: accuracy, precision, recall, and F1 score. We also print sample predictions for each model showing the original tweet, the true label, and the model's predicted label. Finally, we aggregate all results into a comparison table showing accuracy, precision, recall, and F1 side by side across all three models. This lets us directly assess whether social-media-specific pretraining gives BERTweet a measurable advantage over BERT and RoBERTa, which were both pretrained on formal text.